## How to create and use Tools in LangChain

In [2]:
import os 
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:llama-3.3-70b-versatile")
model.invoke("Hello, how are you?")

AIMessage(content="Hello. I'm just a language model, so I don't have feelings or emotions like humans do, but I'm functioning properly and ready to assist you with any questions or topics you'd like to discuss. How can I help you today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 41, 'total_tokens': 91, 'completion_time': 0.107891497, 'completion_tokens_details': None, 'prompt_time': 0.003705888, 'prompt_tokens_details': None, 'queue_time': 0.057487517, 'total_time': 0.111597385}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f9d5a-42b6-7cc1-bdfb-27559b35a6d3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 41, 'output_tokens': 50, 'total_tokens': 91})

In [3]:
from langchain.tools import tool 

@tool
def get_current_weather(location: str) -> str:
    """Get the current weather in a given location.""" #Dockstring helps to describe the function's purpose and usage.
    # This is a placeholder implementation. In a real application, you would call a weather API.
    return f"The current weather in {location} is sunny with a temperature of 25°C."

model_with_tools = model.bind_tools([get_current_weather])


In [6]:
response = model_with_tools.invoke("What's the weather like in New York?")
print(response)

for tool_call in response.tool_calls:
    print(f"Tool : {tool_call['name']}")
    print(f"Input : {tool_call['args']}")


content='' additional_kwargs={'tool_calls': [{'id': 'bzvrj89ss', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_current_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 227, 'total_tokens': 243, 'completion_time': 0.047292484, 'completion_tokens_details': None, 'prompt_time': 0.011566321, 'prompt_tokens_details': None, 'queue_time': 0.058183834, 'total_time': 0.058858805}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019f9d5b-72bb-7ae1-a857-5814b4a22505-0' tool_calls=[{'name': 'get_current_weather', 'args': {'location': 'New York'}, 'id': 'bzvrj89ss', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 227, 'output_tokens': 16, 'total_tokens': 243}
Tool : get_current_weather
Input : {'location': 'New York'}


### Tool Execution loop 

In [10]:
# Model generates tool call

messages = [
    {"role": "user", "content": "What's the weather like in New York?"}
]

ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Execute tool and collect result

for tool_call in ai_msg.tool_calls:
    tool_result = get_current_weather.invoke(tool_call["args"])

    messages.append({
        "role": "tool",
        "tool_call_id": tool_call["id"],
        "content": tool_result,
    })

# Pass result back to the model for the final response

final_response = model_with_tools.invoke(messages)

print(final_response.content)